# 00: Environment and service check

Run this short check before the course notebooks:
1. Ping `agnes-3.0-flash` without exposing credentials.
2. Load one record from the 200-question HotpotQA slice.
3. Open `data/qdrant` and report the collection state.


**Motivation.** Confirm prerequisites without exposing credentials.

**Paper mapping.** This is course infrastructure, not a paper algorithm.

**Next cell.** Resolve imports and report whether each credential exists.

**Failure signals.** Missing flags or imports point to the launcher or kernel environment.

**Read the output.** Both required names should show as SET; their values never appear.

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import check_environment

# Verify environment variables securely
env = check_environment()
print("=== Security & Environment Check ===")
for k, v in env.items():
    status = "SET (Ready)" if v else "MISSING (Required)"
    print(f"  • {k:16}: {status}")

assert env["AGNESAI_API_KEY"], "AGNESAI_API_KEY is missing from environment."
assert env["HF_TOKEN"], "HF_TOKEN is missing from environment."
print("\n[OK] All credentials verified securely in memory.")


=== Security & Environment Check ===
  • AGNESAI_API_KEY : SET (Ready)
  • HF_TOKEN        : SET (Ready)

[OK] All credentials verified securely in memory.


### 1. Ping `agnes-3.0-flash`
Send a small completion request to check network access and model availability. The client reads the API key from the environment and does not log it.


**Motivation.** Separate a working local setup from a working provider connection.

**Paper mapping.** Checks the tutorial model used for evaluation and generation.

**Next cell.** Send one live Chat Completions ping with agnes-3.0-flash.

**Failure signals.** Authentication, rate limits, or an unavailable service stop the request. Never paste a credential into the notebook.

**Read the output.** PONG confirms connectivity. It says nothing about reasoning quality.

In [2]:
from src.agnes_client import chat
from src.config import MODEL_NAME

print(f"Pinging {MODEL_NAME} via Agnes AI Hub...")
response = chat(
    messages=[
        {"role": "user", "content": "Respond strictly with: PONG (Agnes 3.0 Flash Ready)"}
    ],
    model=MODEL_NAME,
    max_tokens=20,
    temperature=0.0,
)

print(f"\nResponse from Agnes AI: {response}")
assert "PONG" in response.upper(), f"Unexpected ping response: {response}"
print("[OK] LLM connectivity verified successfully.")


Pinging agnes-3.0-flash via Agnes AI Hub...



Response from Agnes AI: PONG (Agnes 3.0 Flash Ready)
[OK] LLM connectivity verified successfully.


### 2. Load one HotpotQA row
Check that the Hugging Face slice cache can be read.


**Motivation.** Use real course data.

**Paper mapping.** HotpotQA replaces the paper datasets for this tutorial.

**Next cell.** Load the cached or newly built seed-42 slice and display one row.

**Failure signals.** For HF errors, check HF_TOKEN, the network, and the distractor/validation selection.

**Read the output.** The row includes a question, answer, and annotated supporting titles.

In [3]:
from src.data_hotpot import build_slice

records, smoke_ids = build_slice(n=200, seed=42)
sample = records[0]

print("=== HotpotQA Record Verification ===")
print(f"Slice Total Records: {len(records)}")
print(f"Sample Question ID : {sample['id']}")
print(f"Question           : {sample['question']}")
print(f"Gold Answer        : {sample['gold_answer']}")
print(f"Supporting Titles  : {sample['gold_titles']}")
print(f"Total Paragraphs   : {len(sample['context_paragraphs'])} (Gold + Distractors)")
print("\n[OK] Dataset slice loaded and validated successfully.")


=== HotpotQA Record Verification ===
Slice Total Records: 200
Sample Question ID : 5add1d575542992c1e3a2540
Question           : What nationality was Oliver Reed's character in the film Royal Flash?
Gold Answer        : Prussian
Supporting Titles  : ['Royal Flash (film)', 'Otto von Bismarck']
Total Paragraphs   : 10 (Gold + Distractors)

[OK] Dataset slice loaded and validated successfully.


### 3. Open Qdrant and count the collection
Open embedded Qdrant at `data/qdrant`, report the indexed point count, then close the client.


**Motivation.** Release the embedded database between notebooks.

**Paper mapping.** This is retrieval infrastructure, separate from the paper evaluator.

**Next cell.** Open data/qdrant, report the collection count, and close it.

**Failure signals.** A lock means another process owns this path. Close that kernel instead of deleting the lock.

**Read the output.** No collection is expected before indexing. An existing collection reports its point count.

In [4]:
from src.qdrant_store import get_qdrant_client, close_qdrant_client, DEFAULT_COLLECTION

client = get_qdrant_client()
exists = client.collection_exists(DEFAULT_COLLECTION)

print("=== Embedded Qdrant Vector Store Check ===")
print(f"Collection Name: '{DEFAULT_COLLECTION}'")
print(f"Exists on Disk : {exists}")

if exists:
    info = client.get_collection(DEFAULT_COLLECTION)
    print(f"Indexed Points : {info.points_count}")
else:
    print("Collection not indexed yet. It will be indexed during tutorial execution.")

close_qdrant_client()
print("\n[OK] Qdrant disk store opened and closed cleanly.")


=== Embedded Qdrant Vector Store Check ===
Collection Name: 'hotpot_slice'
Exists on Disk : True
Indexed Points : 1992

[OK] Qdrant disk store opened and closed cleanly.
